# Kidney CT Image Condition Classification — PyTorch

This notebook trains a transfer-learning image classifier using **PyTorch + torchvision VGG16** on local CT images.

### Clinical & Dataset Limitations
- **Diagnostic Scope:** This model classifies anatomical imaging features into four dataset categories (**Normal, Cyst, Stone, Tumor**). It does **not** diagnose or clinically stage Chronic Kidney Disease (CKD).
- **Patient-Level Separation Limitation:** The images in this benchmark dataset are flattened with anonymous labels without DICOM metadata or patient/series identifiers. Therefore, patient-level separation cannot be guaranteed because adjacent CT slices from the same scan or individual may be present.
- **Exact Duplicate Leakage Prevention:** To prevent direct data leakage, this notebook indexes every image by its exact MD5 content hash and groups identical images so that zero identical files appear across the training, validation, and test splits.


In [ ]:
import os
import copy
import random
import hashlib
from pathlib import Path
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Device:', DEVICE)


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IMAGE_SIZE = 224       # VGG16 input dimensions
BATCH_SIZE = 16       # Efficient batch size for CPU execution
EPOCHS = 3          # Epochs for full training
NUM_WORKERS = 2 if os.name != 'nt' else 0

# Set SMOKE_TEST to True for a quick verification run (32 samples/class for 2 epochs)
# Set SMOKE_TEST to False for full production training across the entire dataset
SMOKE_TEST = False
SMOKE_SAMPLES_PER_CLASS = 32


In [ ]:
# Locate the verified local dataset folder
PROJECT_ROOT = Path.cwd()
DATASET_DIR = PROJECT_ROOT / 'CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone'

if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f'Dataset folder not found at {DATASET_DIR.resolve()}')

CLASSES = sorted([d.name for d in DATASET_DIR.iterdir() if d.is_dir() and not d.name.startswith('.')])
CLASS_TO_IDX = {cls_name: i for i, cls_name in enumerate(CLASSES)}

print('Project Root:', PROJECT_ROOT.resolve())
print('Dataset Directory:', DATASET_DIR.resolve())
print('Classes:', CLASSES)
print('Class Mapping:', CLASS_TO_IDX)


In [ ]:
# Group images by content MD5 hash to prevent exact-duplicate leakage across splits
hash_to_info = {}
for cls_name in CLASSES:
    cls_dir = DATASET_DIR / cls_name
    for img_path in sorted(cls_dir.iterdir()):
        if img_path.is_file() and img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            with open(img_path, 'rb') as fp:
                h = hashlib.md5(fp.read()).hexdigest()
            if h not in hash_to_info:
                hash_to_info[h] = {'class': cls_name, 'paths': [img_path]}
            else:
                hash_to_info[h]['paths'].append(img_path)

total_files = sum(len(v['paths']) for v in hash_to_info.values())
unique_images = len(hash_to_info)
print(f'Total dataset files: {total_files}')
print(f'Total unique images (hashes): {unique_images}')
print(f'Exact duplicate copies detected: {total_files - unique_images}')

# Partition hashes into 70% Train, 15% Validation, 15% Test per class
class_to_hashes = defaultdict(list)
for h, info in hash_to_info.items():
    class_to_hashes[info['class']].append(h)

train_samples, val_samples, test_samples = [], [], []
train_hashes, val_hashes, test_hashes = set(), set(), set()
rng = np.random.default_rng(SEED)

print('\n--- Split Statistics (Unique Images per Class) ---')
print(f'{"Class":<10} | {"Total":<8} | {"Train (70%)":<12} | {"Val (15%)":<10} | {"Test (15%)":<10}')
print('-' * 58)

for cls_name in CLASSES:
    cls_h = np.array(sorted(class_to_hashes[cls_name]))
    rng.shuffle(cls_h)
    n = len(cls_h)
    n_train = int(round(0.70 * n))
    n_val = int(round(0.15 * n))
    
    c_train = cls_h[:n_train]
    c_val = cls_h[n_train:n_train + n_val]
    c_test = cls_h[n_train + n_val:]
    
    train_hashes.update(c_train)
    val_hashes.update(c_val)
    test_hashes.update(c_test)
    
    for h in c_train:
        train_samples.append((str(hash_to_info[h]['paths'][0]), CLASS_TO_IDX[cls_name]))
    for h in c_val:
        val_samples.append((str(hash_to_info[h]['paths'][0]), CLASS_TO_IDX[cls_name]))
    for h in c_test:
        test_samples.append((str(hash_to_info[h]['paths'][0]), CLASS_TO_IDX[cls_name]))
        
    print(f'{cls_name:<10} | {n:<8} | {len(c_train):<12} | {len(c_val):<10} | {len(c_test):<10}')

assert len(train_hashes.intersection(val_hashes)) == 0, 'Leakage between train and val!'
assert len(train_hashes.intersection(test_hashes)) == 0, 'Leakage between train and test!'
assert len(val_hashes.intersection(test_hashes)) == 0, 'Leakage between val and test!'
print('\nZero data leakage verified: No duplicate image exists across splits.')
print(f'Total unique splits: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}')


In [ ]:
# Compute class weights for weighted CrossEntropyLoss to handle class imbalance
class_train_counts = [sum(1 for _, label in train_samples if label == i) for i in range(len(CLASSES))]
total_train_samples = len(train_samples)
class_weights = [total_train_samples / (len(CLASSES) * c) for c in class_train_counts]
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print('Training class distribution and loss weights:')
for cls_name, count, w in zip(CLASSES, class_train_counts, class_weights):
    print(f'  {cls_name:<8}: {count} samples | Weight: {w:.4f}')


In [ ]:
# PyTorch Dataset implementation and Transformations
class KidneyCTDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Subsample for smoke testing if enabled
if SMOKE_TEST:
    print(f'SMOKE_TEST is True: Running verification subset ({SMOKE_SAMPLES_PER_CLASS} samples/class)')
    def get_subset(samples, max_per_class):
        grouped = defaultdict(list)
        for s in samples:
            grouped[s[1]].append(s)
        sub = []
        for lbl in sorted(grouped.keys()):
            sub.extend(grouped[lbl][:max_per_class])
        return sub
    active_train = get_subset(train_samples, SMOKE_SAMPLES_PER_CLASS)
    active_val = get_subset(val_samples, max(8, SMOKE_SAMPLES_PER_CLASS // 2))
    active_test = get_subset(test_samples, max(8, SMOKE_SAMPLES_PER_CLASS // 2))
    active_epochs = 2
else:
    print('SMOKE_TEST is False: Running full dataset training')
    active_train = train_samples
    active_val = val_samples
    active_test = test_samples
    active_epochs = EPOCHS

train_dataset = KidneyCTDataset(active_train, transform=train_transform)
val_dataset = KidneyCTDataset(active_val, transform=eval_transform)
test_dataset = KidneyCTDataset(active_test, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print(f'Active DataLoader sizes: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}')


In [ ]:
# Preview one sample image per class
fig, axes = plt.subplots(1, len(CLASSES), figsize=(4 * len(CLASSES), 4))
for ax, cls_name in zip(axes, CLASSES):
    cls_idx = CLASS_TO_IDX[cls_name]
    sample_path = next(p for p, lbl in train_samples if lbl == cls_idx)
    img = Image.open(sample_path).convert('RGB')
    ax.imshow(img)
    ax.set_title(f'{cls_name}\n{Path(sample_path).name}', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Load pretrained VGG16 and configure transfer learning with frozen feature extractor
try:
    weights = models.VGG16_Weights.DEFAULT
    model = models.vgg16(weights=weights)
    print('Loaded pretrained VGG16 ImageNet weights successfully.')
except Exception as exc:
    print('Pretrained weights unavailable; using randomly initialized VGG16:', exc)
    model = models.vgg16(weights=None)

# Freeze convolutional feature extractor
for param in model.features.parameters():
    param.requires_grad = False

# Replace output classification head for 4 classes
in_features = model.classifier[6].in_features
model.classifier[6] = nn.Linear(in_features, len(CLASSES))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
print(model.classifier)


In [ ]:
def run_epoch(loader, training=False):
    model.train(training)
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / max(total, 1), correct / max(total, 1)

best_val_loss = float('inf')
best_state = copy.deepcopy(model.state_dict())
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f'Starting training for {active_epochs} epoch(s)...')
for epoch in range(active_epochs):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)
    scheduler.step(val_loss)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f'Epoch {epoch+1:02d}/{active_epochs:02d} | '
          f'Train loss: {train_loss:.4f}, acc: {train_acc:.4f} | '
          f'Val loss: {val_loss:.4f}, acc: {val_acc:.4f}')
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
print('Training phase complete. Restored best model state.')


In [ ]:
# Plot Training and Validation Loss and Accuracy Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Validation')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# Evaluate on Held-Out Test Set with Classification Report and Confusion Matrix
model.eval()
y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(DEVICE))
        probabilities = torch.softmax(outputs, dim=1).cpu().numpy()
        y_prob.extend(probabilities)
        y_pred.extend(outputs.argmax(dim=1).cpu().numpy())
        y_true.extend(labels.numpy())

y_true = np.asarray(y_true)
y_pred = np.asarray(y_pred)
y_prob = np.asarray(y_prob)

print('Classification Report on Held-Out Test Set:')
print(classification_report(y_true, y_pred, target_names=CLASSES, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASSES)))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Blues')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.title('Confusion Matrix (Held-Out Test Set)')
plt.tight_layout()
plt.show()


In [ ]:
# Save model checkpoint compatible with the prediction application
checkpoint_path = Path('kidney_model.pth')
test_accuracy = float(np.mean(y_true == y_pred))

checkpoint_data = {
    'model_state_dict': model.state_dict(),
    'class_names': CLASSES,
    'class_to_idx': CLASS_TO_IDX,
    'image_size': IMAGE_SIZE,
    'architecture': 'vgg16',
    'mean': [0.485, 0.456, 0.406],
    'std': [0.229, 0.224, 0.225],
    'split_counts': {
        'train': len(train_dataset),
        'val': len(val_dataset),
        'test': len(test_dataset)
    },
    'test_accuracy': test_accuracy,
}

torch.save(checkpoint_data, checkpoint_path)
torch.save(checkpoint_data, 'best_kidney_model.pth')
print('Checkpoint successfully saved:', checkpoint_path.resolve())
print(f'Checkpoint test accuracy: {test_accuracy:.4f}')


In [ ]:
# Predict one local image demonstration
def predict_image(image_path):
    image = Image.open(image_path).convert('RGB')
    tensor = eval_transform(image).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
    order = np.argsort(probs)[::-1]
    return [(CLASSES[i], float(probs[i])) for i in order]

example_path = test_samples[0][0]
example_true_cls = CLASSES[test_samples[0][1]]
print('Testing on local image:', example_path)
print('Ground truth class:', example_true_cls)
predictions = predict_image(example_path)
for cls_name, prob in predictions:
    print(f'  {cls_name:<8}: {prob * 100:.2f}%')
